# Library Mode — Batch Building

`vorpal library` builds every PDF, EPUB, and TXT file in a directory in one command.
It skips files that already have a complete workdir, so re-running is safe and
only processes new or changed files.

This notebook shows:
- Setting up a small library of EPUBs
- Running `vorpal library` across all of them
- Inspecting the output structure
- Using library mode as a pipeline for overnight batch jobs

**Prerequisites:** vorpal installed, ffmpeg on PATH.

In [ ]:
import subprocess
import json
import pathlib
import urllib.request

# Create a small library directory with 3 short Gutenberg EPUBs
library_dir = pathlib.Path("my_library")
library_dir.mkdir(exist_ok=True)

books = [
    (35,   "the_time_machine.epub"),
    (201,  "flatland.epub"),
    (1080, "the_metamorphosis.epub"),
]

for gutenberg_id, filename in books:
    dest = library_dir / filename
    url = f"https://www.gutenberg.org/ebooks/{gutenberg_id}.epub.noimages"
    urllib.request.urlretrieve(url, dest)
    print(f"Downloaded: {dest.name} ({dest.stat().st_size / 1024:.0f} KB)")

print(f"\nLibrary ready: {len(list(library_dir.glob('*.epub')))} EPUBs in {library_dir}/")

## Running `vorpal library`

Pass the directory. All options apply to every build in the batch.

We use `--draft` and `--end-page 20` here to keep this demo fast.
For a real overnight batch job, remove both flags.

In [ ]:
result = subprocess.run(
    [
        "vorpal", "library",
        "--directory", str(library_dir),
        "--voice", "af_heart",
        "--draft",
        "--end-page", "20",
    ],
    capture_output=True, text=True
)
print(result.stdout)
print(f"Return code: {result.returncode}")

## Inspecting the output

Each book gets its own workdir alongside the source file, and a `.m4b` in the library directory.

In [ ]:
print("Library contents:")
for item in sorted(library_dir.iterdir()):
    if item.suffix == ".m4b":
        size_mb = item.stat().st_size / 1e6
        print(f"  {item.name}  ({size_mb:.1f} MB)")
    elif item.is_dir() and item.name.endswith("_workdir"):
        manifest_path = item / "book.json"
        if manifest_path.exists():
            book = json.loads(manifest_path.read_text(encoding="utf-8"))
            chapters = [c for c in book.get("chapters", []) if c.get("include", True)]
            print(f"  {item.name}/  ({len(chapters)} chapters)")

## Re-running is safe

Library mode skips any book that already has a complete workdir. Run it again
and only new or changed files are processed.

In [ ]:
# Add a new book and re-run — only the new book should build
new_book_url = "https://www.gutenberg.org/ebooks/97.epub.noimages"  # War of the Worlds
urllib.request.urlretrieve(new_book_url, library_dir / "war_of_the_worlds.epub")
print("Added: war_of_the_worlds.epub")

result = subprocess.run(
    ["vorpal", "library", "--directory", str(library_dir), "--draft", "--end-page", "20"],
    capture_output=True, text=True
)
print(result.stdout)
print(f"Return code: {result.returncode}")

## Production library job

For an overnight run over a real shelf of books:

```bash
# Build everything in the library at full quality
vorpal library \
  --directory ~/my_audiobooks/ \
  --voice blend_deep_steady
```

On a GPU this takes roughly 25 minutes per book-hour of audio. A shelf of ten
standard novels (~10 hours each) runs overnight without supervision.
The pipeline is resumable — if it's interrupted, the next run picks up from
the last completed chapter of the last incomplete book.

Use `--draft` for a first pass to check chapter detection across the whole
library before committing to full synthesis.